# Test the instruct model interactively

Wraps each instruction in the **same Alpaca template used in training** (this is the key step — the raw `chrono infer` command does *not* do this, so the instruct model would underperform), generates, and shows **only the completion**.

Load the model once (next cell), then iterate on prompts in the cells below without reloading.

**Kernel:** select **Python (chrono)** (Kernel → Change Kernel) so it uses the venv. Verify with the check in the load cell.

In [1]:
import sys
print(sys.executable)  # must be .../persist/venv/bin/python; if not, switch to the 'Python (chrono)' kernel

from chrono_instruct.infer import load, generate, free_memory
from chrono_instruct.data import PROMPT_NO_INPUT, PROMPT_WITH_INPUT

# Local final checkpoint (or an HF repo id). Use the absolute $PERSIST path if that's where it is:
#   REPO = '/home/ubuntu/persist/runs/chrono-instruct-2020/final'
REPO = '../runs/chrono-instruct-2020/final'
model, device = load(REPO)
print('loaded', REPO, 'on', device)

/home/ubuntu/persist/venv/bin/python


/home/ubuntu/persist/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded ../runs/chrono-instruct-2020/final on cuda


In [2]:
def build_prompt(instruction, context=None):
    """Apply the training-time Alpaca template (with-input vs no-input)."""
    instruction = instruction.strip()
    context = (context or '').strip()
    if context:
        return PROMPT_WITH_INPUT.format(instruction=instruction, input=context)
    return PROMPT_NO_INPUT.format(instruction=instruction)

def ask(instruction, context=None, max_new_tokens=200, temperature=0.0, top_k=None,
        repetition_penalty=1.3, no_repeat_ngram_size=3):
    """Return ONLY the completion. Greedy + anti-repetition by default (deterministic AND
    non-looping). Paper-exact greedy: repetition_penalty=1.0, no_repeat_ngram_size=0.
    Varied output: temperature=0.7, top_k=50."""
    prompt = build_prompt(instruction, context)
    return generate(model, device, prompt, max_new_tokens=max_new_tokens,
                    temperature=temperature, top_k=top_k,
                    repetition_penalty=repetition_penalty,
                    no_repeat_ngram_size=no_repeat_ngram_size,
                    return_completion=True).strip()

# Peek at the exact framed prompt the model sees (this IS the Alpaca template from data.py):
print(build_prompt('Explain inflation.'))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Explain inflation.

### Response:



## Quick sanity checks

**Good:** coherent, on-topic, stops cleanly (the model emits `EOT` and halts). **Bad:** echoes the prompt, rambles into a fake new `### Instruction:`, or produces garbage — that points at a format mismatch or an undertrained stage.

In [3]:
tests = [
    'Explain what inflation is in two sentences.',
    'List three risks of holding a single stock instead of a diversified portfolio.',
    'Write a short, professional email declining a meeting invitation.',
]
for t in tests:
    print('=' * 72)
    print('Q:', t)
    print('-' * 72)
    print(ask(t))
    print()

Q: Explain what inflation is in two sentences.
------------------------------------------------------------------------
Inflation refers to changes in prices of goods and services over time, often driven by economic factors such as supply-demand imbalances or technological advancements. It's important to note that this explanation assumes you're referring to something specific within your context rather than general inflation rates across different countries.

Q: List three risks of holding a single stock instead of a diversified portfolio.
------------------------------------------------------------------------
1. **Risk of Excessive Diversification**  
   A single-stock investment can lead to excessive risk if it's too concentrated in one industry or sector, which could result in poor returns and high volatility. This type of investing often involves large investments in stocks with low expected return but also significant risk due to market fluctuations.
 
2. **Lack of Diversity** 


## With an input/context (the `PROMPT_WITH_INPUT` path)

In [4]:
print(ask(
    'Summarize the following text in one sentence.',
    context=('The Federal Reserve raised interest rates by 25 basis points, citing '
             'persistent inflation and a resilient labor market.'),
))

This action was taken to boost economic growth and reduce unemployment, as well as address concerns about inflationary pressures.


## Your own prompt / decoding options

`ask()` defaults to **greedy + anti-repetition** (`repetition_penalty=1.3`, `no_repeat_ngram_size=3`) — deterministic *and* non-looping. Greedy on a ~1.5B model repeats badly without these.

- **Paper-exact greedy** (what the AlpacaEval numbers use): `ask(..., repetition_penalty=1.0, no_repeat_ngram_size=0)` — expect loops.
- **Varied / natural**: `ask(..., temperature=0.7, top_k=50)`.

In [5]:
print(ask('Give three tips for a first-time investor.'))                       # greedy + anti-repetition (default)
# print(ask('Give three tips for a first-time investor.', temperature=0.7, top_k=50))          # sampled
# print(ask('Give three tips for a first-time investor.', repetition_penalty=1.0, no_repeat_ngram_size=0))  # paper-exact greedy (loops)

1. Research your investment opportunity thoroughly before making any decisions. This will help you understand what to look out for and how much money to invest in each asset or portfolio.
2. Be patient with yourself as you navigate through this process, especially if there are unexpected challenges along the way. Remember, it's important to be prepared for setbacks and learn from them rather than letting them discourage you.
3. Don't hesitate to ask questions about different investments during the process of researching. Sometimes, just asking can lead to more informed choices.


## Free the GPU when done (optional)

Run this before loading another vintage in the same kernel, so the two models don't both sit on the card.

In [6]:
# del model
# free_memory()   # gc.collect() + torch.cuda.empty_cache()